In [1]:
import datetime
import numpy as np
import pandas as pd

# ----------------------------------------------------------------
# 0. 실습용 샘플 데이터프레임 생성
# ----------------------------------------------------------------
data = {
    "이름": ["김철수", "이영희", "박민수", "최수연"],
    "가입일자": ["2026-01-01", "2026-05-02", "2026-08-15", "2026-12-25"],
    "나이": [24, 45, 68, 32],
    "신장": [175, 162, 170, 158],  # cm 단위
    "체중": [70, 52, 78, 48],  # kg 단위
    "시험점수": [92, 74, 55, 88],
}

df = pd.DataFrame(data)
print("=== 원본 데이터 ===")
print(df[["이름", "가입일자", "나이", "신장", "체중", "시험점수"]])
print("-" * 50)

=== 원본 데이터 ===
    이름        가입일자  나이   신장  체중  시험점수
0  김철수  2026-01-01  24  175  70    92
1  이영희  2026-05-02  45  162  52    74
2  박민수  2026-08-15  68  170  78    55
3  최수연  2026-12-25  32  158  48    88
--------------------------------------------------


In [5]:
# ----------------------------------------------------------------
# 1. 날짜 데이터 ➡️ 요일, 주말/평일, 계절 파생
# ----------------------------------------------------------------
# 문자열 타입을 datetime 타입으로 변환
df["가입일자"] = pd.to_datetime(df["가입일자"])

# 요일 파생 (0:월, 1:화, ..., 5:토, 6:일)
df["요일"] = df["가입일자"].dt.day_name()  # 영문 요일명 추출

# 주말 여부 파생 (dt.weekday가 5, 6이면 주말, 아니면 평일)
df["주말여부"] = df["가입일자"].dt.weekday.apply(
    lambda x: "주말" if x >= 5 else "평일"
)


# 계절 파생 함수 정의
def get_season(month):
    if month in [3, 4, 5]:
        return "봄"
    elif month in [6, 7, 8]:
        return "여름"
    elif month in [9, 10, 11]:
        return "가을"
    else:
        return "겨울"


df["계절"] = df["가입일자"].dt.month.apply(get_season)

print(df["계절"])

0    겨울
1     봄
2    여름
3    겨울
Name: 계절, dtype: str


In [6]:
# ----------------------------------------------------------------
# 2. 나이 ➡️ 연령대(청년/중년/노년) 파생
# ----------------------------------------------------------------
# pd.cut()을 사용하여 수치형 데이터를 범주형 구간으로 분할
bins = [0, 35, 60, 100]  # 기준 점 (0~35, 35~60, 60~100)
labels = ["청년", "중년", "노년"]
df["연령대"] = pd.cut(df["나이"], bins=bins, labels=labels)

print(df["연령대"] )

0    청년
1    중년
2    노년
3    청년
Name: 연령대, dtype: category
Categories (3, str): ['청년' < '중년' < '노년']


In [7]:
# ----------------------------------------------------------------
# 3. 신장·체중 ➡️ BMI(체질량지수) 계산 (수학적 공식 결합)
# ----------------------------------------------------------------
# 공식: BMI = 체중(kg) / (신장(m) * 신장(m))
df["BMI"] = df["체중"] / ((df["신장"] / 100) ** 2)
df["BMI"] = df["BMI"].round(1)  # 소수점 첫째 자리까지 반올림

print(df["BMI"])

0    22.9
1    19.8
2    27.0
3    19.2
Name: BMI, dtype: float64


In [8]:
# ----------------------------------------------------------------
# 4. 점수 합산 ➡️ 등급(A/B/C) 파생
# ----------------------------------------------------------------
# np.select()를 사용하면 여러 조건에 따른 다중 분기가 깔끔해집니다.
conditions = [
    df["시험점수"] >= 90,  # A 조건
    df["시험점수"] >= 70,  # B 조건
    df["시험점수"] < 70,  # C 조건
]
grade_labels = ["A", "B", "C"]

df["등급"] = np.select(conditions, grade_labels, default="F")

print(df["등급"])

0    A
1    B
2    C
3    B
Name: 등급, dtype: str


In [9]:
# ----------------------------------------------------------------
# 5. Feature Engineering 최종 결과 확인
# ----------------------------------------------------------------
print("\n=== Feature Engineering 적용 후 데이터 ===")
columns_to_show = [
    "이름",
    "요일",
    "주말여부",
    "계절",
    "연령대",
    "BMI",
    "등급",
]
print(df[columns_to_show])


=== Feature Engineering 적용 후 데이터 ===
    이름        요일 주말여부  계절 연령대   BMI 등급
0  김철수  Thursday   평일  겨울  청년  22.9  A
1  이영희  Saturday   주말   봄  중년  19.8  B
2  박민수  Saturday   주말  여름  노년  27.0  C
3  최수연    Friday   평일  겨울  청년  19.2  B
